In [30]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, f1_score

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import joblib

In [31]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("naserabdullahalam/phishing-email-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/naserabdullahalam/phishing-email-dataset


In [32]:
import pandas as pd
import os

# Download dataset
path = kagglehub.dataset_download(
    "naserabdullahalam/phishing-email-dataset"
)

print(path)

# See files
print(os.listdir(path))

/kaggle/input/datasets/naserabdullahalam/phishing-email-dataset
['SpamAssasin.csv', 'Nazario.csv', 'Nigerian_Fraud.csv', 'CEAS_08.csv', 'Enron.csv', 'Ling.csv', 'phishing_email.csv']


In [33]:
import pandas as pd

df = pd.read_csv(
    "/kaggle/input/datasets/naserabdullahalam/phishing-email-dataset/phishing_email.csv"
)

df.head()

,text_combined,label
0,hpl nom may 25 2001 see attached file hplno 52...,0
1,nom actual vols 24 th forwarded sabrae zajac h...,0
2,enron actuals march 30 april 1 201 estimated a...,0
3,hpl nom may 30 2001 see attached file hplno 53...,0
4,hpl nom june 1 2001 see attached file hplno 60...,0


In [34]:
df.describe()

,label
count,82486.000000
mean,0.519979
std,0.499604
min,0.000000
25%,0.000000
50%,1.000000
75%,1.000000
max,1.000000


In [35]:
df.isna().sum()

text_combined    0
label            0
dtype: int64

In [36]:
df.duplicated().sum()

np.int64(408)

In [37]:
df = df.drop_duplicates()

In [38]:
df.duplicated().sum()

np.int64(0)

In [40]:
df.isna().sum()

text_combined    0
label            0
dtype: int64

In [42]:
X = df.drop("label", axis = 1)

In [43]:
y = df["label"]

In [44]:
df["text_combined"].str.len().min()

1

In [45]:
y.value_counts()

label
1    42845
0    39233
Name: count, dtype: int64

In [53]:
X = df["text_combined"]
y = df["label"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [54]:

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [55]:
print(X_train_tfidf.shape)
print(y_train.shape)

(65662, 5000)
(65662,)


In [57]:
models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        eval_metric="logloss",
        random_state=42
    )
}

In [58]:
results = {}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    preds = model.predict(X_test_tfidf)

    f1 = f1_score(y_test, preds)
    roc = roc_auc_score(y_test, preds)

    results[name] = {
        "F1 Score": f1,
        "ROC AUC": roc
    }

    print(f"\n{name}")
    print(classification_report(y_test, preds))


Logistic Regression
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      7847
           1       0.98      0.98      0.98      8569

    accuracy                           0.98     16416
   macro avg       0.98      0.98      0.98     16416
weighted avg       0.98      0.98      0.98     16416


Random Forest
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      7847
           1       0.99      0.99      0.99      8569

    accuracy                           0.99     16416
   macro avg       0.99      0.99      0.99     16416
weighted avg       0.99      0.99      0.99     16416


XGBoost
              precision    recall  f1-score   support

           0       0.99      0.97      0.98      7847
           1       0.97      0.99      0.98      8569

    accuracy                           0.98     16416
   macro avg       0.98      0.98      0.98     16416
weighted avg       0.98      

In [60]:
results_df = pd.DataFrame(results).T
print(results_df.sort_values("F1 Score", ascending=False))

                     F1 Score   ROC AUC
Random Forest        0.987653  0.986975
Logistic Regression  0.979977  0.978891
XGBoost              0.979127  0.977545


In [62]:
best_model = RandomForestClassifier(n_estimators = 100, random_state=42)
best_model.fit(X_train_tfidf, y_train)

RandomForestClassifier(random_state=42)

In [65]:
preds = best_model.predict(X_test_tfidf)
print(classification_report(y_test, preds))


              precision    recall  f1-score   support

           0       0.99      0.98      0.99      7847
           1       0.98      0.99      0.99      8569

    accuracy                           0.99     16416
   macro avg       0.99      0.99      0.99     16416
weighted avg       0.99      0.99      0.99     16416



In [66]:
roc_auc_score(y_test, model.predict_proba(X_test_tfidf)[:,1])

np.float64(0.9979344281355481)

In [67]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, preds))

[[7717  130]
 [  94 8475]]


In [74]:
joblib.dump(best_model, "phishing_model.pkl")
joblib.dump(tfidf, "tfidf.pkl")

['tfidf.pkl']

In [75]:
email = ["Urgent! Your bank account has been suspended. Click the link to verify immediately."]

email_tfidf = tfidf.transform(email)
prediction = model.predict(email_tfidf)

print("Prediction:", prediction[0])

Prediction: 1


In [72]:
import os

print(os.listdir())

['phishing_model.pkl', 'tfidf.pkl', '.virtual_documents']
